<a href="https://colab.research.google.com/github/SomanMasood/Data-Analysis-Project-EDA-/blob/main/Joins(SQL).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')

cursor = conn.cursor()

# PHASE 4: Joins (The Heart of SQL)

In [ ]:
cursor.execute(
    """
    CREATE TABLE Departments
    (
    dept_id INTEGER PRIMARY KEY,
    dept_name TEXT
    )
    """

)

conn.commit()

In [ ]:
cursor.execute(
    """
    CREATE TABLE Employees
    (
    emp_id INTEGER PRIMARY KEY,
    emp_name TEXT,
    dept_id INTEGER,
    FOREIGN KEY(dept_id)
    REFERENCES Departments(dept_id)
    )
    """
)

conn.commit()

In [ ]:
cursor.executemany(
    """
    INSERT INTO Departments
    VALUES (?, ?)
    """ ,
    [
      (101, 'IT'),
    (102, 'HR'),
    (103, 'Finance')
    ]
)

conn.commit()

In [ ]:
cursor.executemany(
    """
    INSERT INTO Employees
    VALUES (?, ?, ?)
    """,
    [
    (1, 'John', 101),
    (2, 'Sarah', 102),
    (3, 'Mike', 101)
    ]
)

conn.commit()

In [ ]:
query = """
SELECT *
FROM Employees
"""

df = pd.read_sql(query , conn)

print(df)

   emp_id emp_name  dept_id
0       1     John      101
1       2    Sarah      102
2       3     Mike      101


In [ ]:
query = """
SELECT *
FROM Departments
"""

df = pd.read_sql(query , conn)

print(df)

   dept_id dept_name
0      101        IT
1      102        HR
2      103   Finance


In [ ]:
query = """
SELECT *
FROM Employees
INNER JOIN Departments
ON Employees.dept_id = Departments.dept_id
"""

df = pd.read_sql(query , conn)

print(df)

   emp_id emp_name  dept_id  dept_id dept_name
0       1     John      101      101        IT
1       2    Sarah      102      102        HR
2       3     Mike      101      101        IT


Definition of INNER JOIN

`Return only rows that match in both tables.`

Memorize that.

Everything else is details.

query :

```
FROM A
INNER JOIN B
ON A.id = B.id
```
Read it as:

`Take table A, take table B, and keep only rows where the IDs match.`


In [ ]:
# Cleaner Query

query = """
SELECT
E.emp_name , D.dept_name
FROM Employees E
INNER JOIN Departments D
ON E.dept_id = D.dept_id
"""

df = pd.read_sql(query , conn)

print(df)

  emp_name dept_name
0     John        IT
1    Sarah        HR
2     Mike        IT


What does ON do?

`ON E.department_id = D.department_id`

Answer:

Specifies the matching condition
between the two tables.

Finance does not appear because there is no employee whose department_id = 103, and INNER JOIN returns only rows that have matching values in both tables.

Think of INNER JOIN as a VIP party:

`Employees Table  ∩  Departments Table`

Only rows that exist on both sides get through the door.

**LEFT JOIN**

This is arguably the most useful join for analysts.

Definition

```
Return ALL rows from the LEFT table
+
Matching rows from the RIGHT table

```


If no match exists on the right side:

NULL

is filled in.

In [ ]:
cursor.execute(
    """
    INSERT INTO Employees
    VALUES (4, 'Alex', 999)
    """
)

conn.commit()

In [ ]:
query = """
SELECT
E.emp_name , D.dept_name
FROM Employees E
INNER JOIN Departments D
ON E.dept_id = D.dept_id
"""

df = pd.read_sql(query , conn)

print(df)

  emp_name dept_name
0     John        IT
1    Sarah        HR
2     Mike        IT


In [ ]:
query = """
SELECT
E.emp_name , D.dept_name
FROM Employees E
LEFT JOIN Departments D
ON E.dept_id = D.dept_id
"""

df = pd.read_sql(query , conn)

print(df)

  emp_name dept_name
0     John        IT
1    Sarah        HR
2     Mike        IT
3     Alex      None


Most Common Analyst Use Case

Finding missing data.

Business meaning:

Show employees assigned to invalid or missing departments.

In [ ]:
query = """
SELECT
E.emp_name , D.dept_name
FROM Employees E
LEFT JOIN Departments D
ON E.dept_id = D.dept_id
WHERE D.dept_id IS NULL
"""

df = pd.read_sql(query , conn)

print(df)

  emp_name dept_name
0     Alex      None


RIGHT JOIN

f We Want ALL Departments

Conceptually we want:

Keep every row from Departments

This is what a RIGHT JOIN does.

Conceptual RIGHT JOIN
```

SELECT
    E.name,
    D.department_name
FROM Employees E
RIGHT JOIN Departments D
ON E.department_id = D.department_id;
```

But SQLite Doesn't Support RIGHT JOIN

If you try:

RIGHT JOIN

SQLite throws an error.

SQLite Solution

Swap the tables and use LEFT JOIN.

because:


```

RIGHT JOIN
=
LEFT JOIN with the tables reversed
```



In [ ]:
query = """
SELECT
E.emp_name , D.dept_name
FROM Employees E
RIGHT JOIN Departmests D
ON E.dept_id = D.dept_id
"""

df = pd.read_sql(query , conn)

print(df)

DatabaseError: Execution failed on sql ' 
SELECT 
E.emp_name , D.dept_name 
FROM Employees E
RIGHT JOIN Departmests D
ON E.dept_id = D.dept_id
': RIGHT and FULL OUTER JOINs are not currently supported

In [ ]:
query = """
SELECT
D.dept_name , E.emp_name
FROM Departments D
LEFT JOIN Employees E
ON E.dept_id = D.dept_id
"""

df = pd.read_sql(query , conn)

print(df)

  dept_name emp_name
0        IT     John
1        IT     Mike
2        HR    Sarah
3   Finance     None


The table on the left is guaranteed to keep all of its rows.

That single sentence is the heart of LEFT/RIGHT JOINs.

**FULL OUTER JOIN**

Definition

Return ALL rows from both tables.

If a row has no match:

Fill the missing side with NULL.

Think of it as:

```
LEFT JOIN
+
RIGHT JOIN
```
combined into one result.

In [ ]:
query = """
SELECT
E.emp_name , D.dept_name
FROM Employees E
FULL OUTER JOIN Departments D
ON E.dept_id = D.dept_id
"""

df = pd.read_sql(query , conn)

print(df)

DatabaseError: Execution failed on sql '
SELECT 
E.emp_name , D.dept_name
FROM Employees E
FULL OUTER JOIN Departments D
ON E.dept_id = D.dept_id 
': RIGHT and FULL OUTER JOINs are not currently supported

Visual Model

Imagine two circles.

INNER JOIN

Only overlap

`(A ∩ B)`

LEFT JOIN

```
Entire left circle
+
overlap
```
RIGHT JOIN

```
Entire right circle
+
overlap
```


FULL OUTER JOIN

```

Everything

A ∪ B

```


No row is discarded.

**Self Join**

A Self Join means:

Employees ↔ Employees

The same table joins with itself.

At first it feels strange, but it solves many real-world problems.

In [ ]:
cursor.execute("""
CREATE TABLE Employee_Manager
(
    emp_id INTEGER PRIMARY KEY,
    name TEXT,
    manager_id INTEGER
)
""")

cursor.executemany("""
INSERT INTO Employee_Manager
VALUES (?, ?, ?)
""", [
    (1, 'CEO', None),
    (2, 'Alice', 1),
    (3, 'Bob', 1),
    (4, 'Charlie', 2)
])

conn.commit()

In [ ]:
query = """ SELECT *
FROM Employee_Manager """

df = pd.read_sql(query , conn)

print(df)

   emp_id     name  manager_id
0       1      CEO         NaN
1       2    Alice         1.0
2       3      Bob         1.0
3       4  Charlie         2.0


Business Question

Show each employee along with their manager's name.

| Employee | Manager |
| -------- | ------- |
| Alice    | CEO     |
| Bob      | CEO     |
| Charlie  | Alice   |


In [ ]:
query = """ SELECT
    E.name AS Employee,
    M.name AS Manager
FROM Employee_Manager E
JOIN Employee_Manager M
ON E.manager_id = M.emp_id """

df = pd.read_sql(query , conn)

print(df)

  Employee Manager
0    Alice     CEO
1      Bob     CEO
2  Charlie   Alice


Notice:

```
Employee_Manager E
Employee_Manager M
```
Same table.

Two aliases.

We're pretending there are two copies:

```
E = Employee side
M = Manager side
```



# Join Logic Deep Dive

How Does a JOIN Actually Work?

internally, a join can be understood as:

```
Step 1: Create combinations
Step 2: Apply matching condition
Step 3: Keep rows according to join type
```

Cartesian Product

Formula

If:

```
Table A has m rows
Table B has n rows

```


Then:

 Cartesian Product
=
m × n rows

Examples:


```

3 × 4 = 12
10 × 5 = 50
100 × 100 = 10,000
```
Why Do We Care?

Because an INNER JOIN can be thought of as:

Cartesian Product
+
Filter


In [ ]:
cursor.execute(
    """
    CREATE TABLE Customers (
    customer_id INTEGER PRIMARY KEY,
    customer name TEXT
    )
    """
)

cursor.execute(
    """
    CREATE TABLE Orders
    (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER
    )
    """
)

cursor.execute(
    """
    CREATE TABLE Order_items
    (
     order_id INTEGER,
     product_id INTEGER
    )
    """
)

conn.commit()

In [ ]:
cursor.execute("""
CREATE TABLE Products
(
    product_id INTEGER PRIMARY KEY,
    product_name TEXT
)
""")

conn.commit()

In [ ]:
cursor.executemany(
    """
    INSERT INTO Customers
    VALUES (? , ?) """ ,
    [
        (1, "Alice"),
        (2, "Bob")
    ]
)

cursor.executemany(
    """
    INSERT INTO Orders
    VALUES (? , ?) """ ,
    [
        (101, 1),
        (102, 1),
        (103, 2)
    ]
)


cursor.executemany(
    """
    INSERT INTO Products
    VALUES (?, ?) """ ,
    [
        (10, "Laptop"),
        (20, "Phone")
    ]
)
cursor.executemany(
    """
    INSERT INTO Order_items
    VALUES (? , ?) """ ,
    [
        (101, 10),
        (101, 20),
        (102, 20),
        (103, 10)
    ]
)

conn.commit()

In [ ]:
query = """
SELECT *
FROM Customers
"""

df = pd.read_sql(query , conn)

print(df)

query = """
SELECT *
FROM Orders
"""

df1 = pd.read_sql(query , conn)

print(df1)

query = """
SELECT *
FROM Products
"""

df3 = pd.read_sql(query , conn)

print(df3)

query = """
SELECT *
FROM Order_items
"""

df2 = pd.read_sql(query , conn)

print(df2)

   customer_id customer
0            1    Alice
1            2      Bob
   order_id  customer_id
0       101            1
1       102            1
2       103            2
   product_id product_name
0          10       Laptop
1          20        Phone
   order_id  product_id
0       101          10
1       101          20
2       102          20
3       103          10


In [ ]:
query = """
SELECT *
FROM Customers C
JOIN Orders O
ON C.customer_id = O.customer_id
"""

df = pd.read_sql(query , conn)

print(df)

   customer_id customer  order_id  customer_id
0            1    Alice       101            1
1            1    Alice       102            1
2            2      Bob       103            2


In [ ]:
query = """
SELECT
C.customer, O.order_id , OI.product_id
FROM Customers C
JOIN Orders O
ON C.customer_id = O.customer_id
JOIN Order_items OI
ON O.order_id = OI.order_id
"""

df = pd.read_sql(query , conn)

print(df)

  customer  order_id  product_id
0    Alice       101          10
1    Alice       101          20
2    Alice       102          20
3      Bob       103          10


In [ ]:
query = """
SELECT
C.customer, P.product_name
FROM Customers C
JOIN Orders O
ON C.customer_id = O.customer_id
JOIN Order_items OI
ON O.order_id = OI.order_id
JOIN Products P
ON OI.product_id = P.product_id
"""

df = pd.read_sql(query , conn)

print(df)

  customer product_name
0    Alice       Laptop
1    Alice        Phone
2    Alice        Phone
3      Bob       Laptop


# Aggregation + Joins

Problem 1: Number of Orders Per Customer

Business asks:

How many orders has each customer placed?

In [ ]:
query = """ SELECT
C.customer AS name,
COUNT(O.order_id) AS num_orders
FROM Customers C
JOIN Orders O
ON C.customer_id = O.customer_id
GROUP BY C.customer_id
"""

df = pd.read_sql(query , conn)

print(df)


    name  num_orders
0  Alice           2
1    Bob           1




```
Join Customers and Orders
↓
Group by customer
↓
Count orders
```



Problem 2: Products Purchased By Each Customer